# Conventional baseline vs. the PLEM joint loss, scored both ways

This notebook pairs two executions of `train_unet_joint_scaled.ipynb` on **the same test tiles**:

- **`baseline`** (`PLEM_VARIANT=baseline`): conventional training. Cross-entropy + soft Dice on
  road/building, road intersections as an ordinary BCE + Dice mask decoded by connected
  components, and the checkpoint chosen by validation mIoU (`losses/baseline.py`).
- **`plem`**: the joint loss (`PLEMMultiTaskLoss`, independent heatmap point head), as in run 6.

Everything else is identical: data, tile split, patches, building-label masking, per-source class
masking, dark augmentation, seed, `SmallUNet`, epoch budget and evaluation code.

Each test tile is scored with **conventional metrics** (pixel accuracy, IoU, mIoU) and **PLEM's
metrics** (`cbhm`, `cbhm_soft`, `dtaf1`, `dtaf1_topo`, road APLS, BF, clDice, intersection F1). Two
separate questions:

1. **Metric claim.** Do IoU and PLEM's metrics disagree about these two models, overall or tile by
   tile? Where they do, the panels below show which one matches what the prediction looks like.
2. **Loss claim.** At matched training, does the joint loss score higher on PLEM's metrics?

Neither outcome is assumed; the closing cell records whatever the numbers show.

**Inputs:** `results/scaled/{baseline,plem}_{test,dark_test,val_calibration}.csv` and
`*_run_info.json` (tracked in git), plus the per-tile prediction cache in
`results/scaled/preds/` and the image cache in `data/` for the side-by-side panels (only on the
machine that ran the two jobs; the panels are skipped elsewhere). CPU only.

In [ ]:
import json
import sys, os
from pathlib import Path
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, kendalltau

RESULTS_DIR = Path(os.environ.get("PLEM_RESULTS_DIR", Path("..").resolve() / "results" / "scaled"))
DATA_DIR = Path("..").resolve() / "data"
VARIANTS = ["baseline", "plem"]
LABELS = {"baseline": "Conventional (CE+Dice, mIoU-selected)", "plem": "PLEM joint loss"}
# Fixed entity colours (categorical slots 2 and 1), used in every figure.
COLORS = {"baseline": "#eb6834", "plem": "#2a78d6"}


def _load(kind):
    out = {}
    for v in VARIANTS:
        p = RESULTS_DIR / f"{v}_{kind}.csv"
        if not p.exists():
            raise FileNotFoundError(f"{p} is missing -- run train_unet_joint_scaled.ipynb with "
                                    f"PLEM_VARIANT={v} (RESCORE_ONLY=True is enough for 'plem').")
        out[v] = pd.read_csv(p)
    return out


test = _load("test")
dark = _load("dark_test")
run_info = {v: json.load(open(RESULTS_DIR / f"{v}_run_info.json")) for v in VARIANTS}
for v in VARIANTS:
    print(f"{v:9s} {run_info[v]}")

# Pair by tile. Both runs must have scored exactly the same test tiles.
assert set(test["baseline"]["tile"]) == set(test["plem"]["tile"]), "test tile sets differ"
paired = test["baseline"].merge(test["plem"], on=["tile", "source"], suffixes=("_b", "_p"))
print("\npaired test tiles per source:", paired["source"].value_counts().to_dict())

## 1. Headline table

Means per source. Segmentation metrics first (conventional, then PLEM's), then intersections
micro-averaged over SpaceNet test tiles. The baseline's intersections are shown twice: at its
conventional 0.5 threshold, and at a threshold calibrated on validation tiles exactly as the joint
model's is, so a point gap cannot be put down to the threshold.

In [ ]:
CONV = ["pixel_acc", "miou", "iou_road", "iou_building"]
PLEM = ["cbhm", "cbhm_soft", "dtaf1", "dtaf1_topo", "road_apls", "cldice_mean", "bf_mean"]
SEG_METRICS = CONV + PLEM
ROAD_ONLY = {"iou_road", "road_apls", "cldice_mean"}

seg_table = pd.concat({LABELS[v]: test[v].groupby("source")[SEG_METRICS].mean() for v in VARIANTS},
                      names=["model"]).swaplevel().sort_index()
display(seg_table.round(3))


def micro(df, suffix=""):
    tp, n_pred, n_gt = (df[f"n_tp_points{suffix}"].sum(), df[f"n_pred_points{suffix}"].sum(),
                        df[f"n_gt_points{suffix}"].sum())
    p = tp / n_pred if n_pred else 0.0
    r = tp / n_gt if n_gt else 0.0
    return {"predicted": int(n_pred), "real": int(n_gt), "matched": int(tp),
            "pred_per_real": n_pred / max(n_gt, 1), "precision": p, "recall": r,
            "f1": 2 * p * r / (p + r) if p + r else 0.0,
            "pixel_iou_mean": df[f"iou_point{suffix}"].mean()}


point_rows = {}
for v in VARIANTS:
    sn = test[v][test[v]["source"] == "spacenet"]
    info = run_info[v]
    point_rows[f"{LABELS[v]} @ default {info['point_thresh_default']}"] = micro(sn, "_default_thresh")
    point_rows[f"{LABELS[v]} @ calibrated {info['point_thresh']}"] = micro(sn)
point_table = pd.DataFrame(point_rows).T
print("Road intersections, SpaceNet test tiles (micro-averaged):")
display(point_table.round(3))

## 2. Paired per-tile differences (PLEM minus baseline)

The tile is the unit: each test tile was predicted by both models. For each metric: the mean
difference, a 95% bootstrap confidence interval (10,000 resamples of tiles) and a two-sided
Wilcoxon signed-rank p-value. A positive difference favours the joint loss. **The question for the
metric claim is whether the conventional and PLEM metrics point the same way.**

In [ ]:
rng = np.random.default_rng(0)
N_BOOT = 10_000


def paired_stats(d):
    d = d[np.isfinite(d)]
    if len(d) < 2:
        return dict(n=len(d), mean=np.nan, ci_lo=np.nan, ci_hi=np.nan, p=np.nan, plem_wins=np.nan)
    boots = rng.choice(d, (N_BOOT, len(d)), replace=True).mean(axis=1)
    p = wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    return dict(n=len(d), mean=d.mean(), ci_lo=np.percentile(boots, 2.5),
                ci_hi=np.percentile(boots, 97.5), p=p, plem_wins=float((d > 0).mean()))


diff_rows = []
for source, g in paired.groupby("source"):
    # Road-only metrics are undefined on the road-less sources (Potsdam, SpaceNet6).
    metrics = (SEG_METRICS + ["point_f1"] if source == "spacenet"
               else [m for m in SEG_METRICS if m not in ROAD_ONLY])
    for m in metrics:
        d = (g[f"{m}_p"] - g[f"{m}_b"]).to_numpy(dtype=float)
        kind = "conventional" if m in CONV else "PLEM"
        diff_rows.append({"source": source, "metric": m, "kind": kind, **paired_stats(d)})
diffs = pd.DataFrame(diff_rows)
diffs["significant"] = (diffs["ci_lo"] > 0) | (diffs["ci_hi"] < 0)
display(diffs.round(4))
diffs.to_csv(RESULTS_DIR / "comparison_paired_diffs.csv", index=False)

In [ ]:
# One panel per source: mean paired difference with its 95% bootstrap CI, conventional metrics on
# top, PLEM's below. Zero = no difference; right of zero favours the joint loss.
sources = sorted(diffs["source"].unique())
fig, axes = plt.subplots(1, len(sources), figsize=(5.2 * len(sources), 5), sharex=False)
axes = np.atleast_1d(axes)
for ax, source in zip(axes, sources):
    sub = diffs[(diffs["source"] == source) & diffs["mean"].notna()].reset_index(drop=True)
    y = np.arange(len(sub))[::-1]
    for yi, (_, r) in zip(y, sub.iterrows()):
        color = "#52514e" if r["kind"] == "conventional" else "#2a78d6"
        ax.plot([r["ci_lo"], r["ci_hi"]], [yi, yi], color=color, linewidth=2)
        ax.plot(r["mean"], yi, "o", color=color, markersize=8,
                markeredgecolor="white", markeredgewidth=2)
    ax.axvline(0, color="#a3a29b", linewidth=1)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{m}  ({k[:4]})" for m, k in zip(sub["metric"], sub["kind"])])
    ax.set_title(source)
    ax.set_xlabel("PLEM - baseline (mean, 95% CI)")
    ax.grid(axis="x", color="#e4e3dd", linewidth=0.8)
    ax.set_axisbelow(True)
fig.suptitle("Paired per-tile differences: grey = conventional metric, blue = PLEM metric")
fig.tight_layout()
plt.show()

## 3. Do IoU and PLEM's metrics agree tile by tile?

Two views on SpaceNet (the only source with roads and intersections):

- **Rank agreement within each model:** Kendall's tau between per-tile mIoU and each PLEM metric.
  A low tau means the two kinds of metric order the same predictions differently.
- **Disagreement between models:** tiles where the change in mIoU and the change in `dtaf1_topo`
  (or `cbhm`) have opposite signs, i.e. the conventional metric prefers one model and PLEM's the
  other. The largest of these are drawn side by side in section 4.

In [ ]:
tau_rows = []
for v in VARIANTS:
    sn = test[v][test[v]["source"] == "spacenet"]
    for m in PLEM + ["point_f1"]:
        ok = sn[["miou", m]].dropna()
        t, p = kendalltau(ok["miou"], ok[m]) if len(ok) > 2 else (np.nan, np.nan)
        tau_rows.append({"model": v, "metric": m, "kendall_tau_vs_miou": t, "p": p, "n": len(ok)})
display(pd.DataFrame(tau_rows).pivot(index="metric", columns="model", values="kendall_tau_vs_miou").round(3))

sn_pair = paired[paired["source"] == "spacenet"].copy()
for m in ["miou", "dtaf1_topo", "cbhm", "road_apls", "point_f1"]:
    sn_pair[f"d_{m}"] = sn_pair[f"{m}_p"] - sn_pair[f"{m}_b"]
for m in ["dtaf1_topo", "cbhm"]:
    opp = np.sign(sn_pair["d_miou"]) * np.sign(sn_pair[f"d_{m}"]) < 0
    print(f"tiles where d_mIoU and d_{m} disagree in sign: {int(opp.sum())}/{len(sn_pair)}")

sn_pair["disagreement"] = np.where(np.sign(sn_pair["d_miou"]) * np.sign(sn_pair["d_dtaf1_topo"]) < 0,
                                   (sn_pair["d_dtaf1_topo"] - sn_pair["d_miou"]).abs(), 0.0)
disagree = sn_pair.sort_values("disagreement", ascending=False)
display(disagree[disagree["disagreement"] > 0]
        [["tile", "d_miou", "d_dtaf1_topo", "d_cbhm", "d_road_apls", "d_point_f1",
          "miou_b", "miou_p", "dtaf1_topo_b", "dtaf1_topo_p"]].head(10).round(3))

In [ ]:
# Per-tile change in mIoU against change in DTAF1-Topo. Points off the diagonal quadrants
# (top-left, bottom-right) are tiles where the two kinds of metric prefer different models.
fig, ax = plt.subplots(figsize=(6, 6))
ax.axhline(0, color="#a3a29b", linewidth=1)
ax.axvline(0, color="#a3a29b", linewidth=1)
ax.scatter(sn_pair["d_miou"], sn_pair["d_dtaf1_topo"], s=40, color="#2a78d6",
           edgecolor="white", linewidth=1.5)
for _, r in disagree[disagree["disagreement"] > 0].head(4).iterrows():
    ax.annotate(r["tile"], (r["d_miou"], r["d_dtaf1_topo"]), fontsize=8, color="#52514e",
                xytext=(4, 4), textcoords="offset points")
lim = np.nanmax(np.abs(sn_pair[["d_miou", "d_dtaf1_topo"]].to_numpy())) * 1.1 or 1.0
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_xlabel("change in mIoU (PLEM - baseline)")
ax.set_ylabel("change in DTAF1-Topo (PLEM - baseline)")
ax.set_title("SpaceNet test tiles: do the two metrics prefer the same model?")
ax.grid(color="#e4e3dd", linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

## 4. Side by side: the tiles where the metrics disagree most

Image, ground truth, baseline and joint-model predictions (road red, building green, intersections
yellow; blue-grey = no public building labels). Needs `data/` and the prediction cache, so this
cell only draws on the machine that ran both jobs.

In [ ]:
import torch
import torch.nn.functional as F


def overlay(label, point_mask=None, building_valid=None, point_radius=6):
    o = np.zeros((*label.shape, 3), dtype=np.uint8)
    if building_valid is not None:
        o[building_valid == 0] = [70, 90, 150]
    o[label == 1] = [255, 0, 0]
    o[label == 2] = [0, 255, 0]
    if point_mask is not None and point_mask.any():
        m = torch.from_numpy(point_mask.astype(np.float32))[None, None]
        big = F.max_pool2d(m, 2 * point_radius + 1, stride=1, padding=point_radius)[0, 0].numpy() > 0
        o[big] = [255, 255, 0]
    return o


def tile_npz(tile, source):
    hits = list((DATA_DIR / source).rglob(f"{tile}.npz")) if (DATA_DIR / source).is_dir() else []
    return np.load(hits[0]) if hits else None


def side_by_side(tiles, title):
    from datasets.intersections import points_to_mask
    rows = []
    for tile in tiles:
        src = paired.loc[paired["tile"] == tile, "source"].iloc[0]
        gt = tile_npz(tile, src)
        preds = {v: RESULTS_DIR / "preds" / v / f"{tile}.npz" for v in VARIANTS}
        if gt is None or not all(p.exists() for p in preds.values()):
            continue
        rows.append((tile, src, gt, {v: np.load(p) for v, p in preds.items()}))
    if not rows:
        print(f"({title}: image or prediction cache not available here -- skipped)")
        return
    fig, axes = plt.subplots(len(rows), 4, figsize=(18, 4.6 * len(rows)))
    axes = np.atleast_2d(axes)
    for ax_row, (tile, src, gt, pr) in zip(axes, rows):
        valid = gt["building_valid"] if "building_valid" in gt.files else None
        pts = (points_to_mask(gt["points"], gt["label"].shape, radius=2)
               if "points" in gt.files else None)
        s = paired.loc[paired["tile"] == tile].iloc[0]
        ax_row[0].imshow(gt["image"])
        ax_row[0].set_title(f"{tile} ({src})")
        ax_row[1].imshow(gt["image"])
        ax_row[1].imshow(overlay(np.where(gt["label"] == 3, 0, gt["label"]), pts, valid), alpha=0.5)
        ax_row[1].set_title("ground truth")
        for ax, v, suf in ((ax_row[2], "baseline", "_b"), (ax_row[3], "plem", "_p")):
            ax.imshow(gt["image"])
            ax.imshow(overlay(pr[v]["seg"], pr[v]["points"]), alpha=0.5)
            ax.set_title(f"{v}: mIoU {s['miou' + suf]:.2f}  DTAF1-Topo {s['dtaf1_topo' + suf]:.2f}\n"
                         f"APLS {s['road_apls' + suf]:.2f}  BF {s['bf_mean' + suf]:.2f}  "
                         f"pointF1 {s['point_f1' + suf]:.2f}")
        for ax in ax_row:
            ax.axis("off")
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    plt.show()


side_by_side(disagree[disagree["disagreement"] > 0]["tile"].head(4).tolist(),
             "Largest mIoU-vs-DTAF1-Topo disagreements (SpaceNet)")
side_by_side(sn_pair.sort_values("d_point_f1", ascending=False)["tile"].head(2).tolist(),
             "Largest intersection-F1 gains for the joint loss")

## 5. Darkened input: both models on the same severity sweep

In [ ]:
DARK_METRICS = ["miou", "cbhm_soft", "dtaf1_topo", "point_f1"]
sources = sorted(dark["plem"]["source"].unique())
fig, axes = plt.subplots(len(sources), len(DARK_METRICS), figsize=(4 * len(DARK_METRICS), 3.3 * len(sources)),
                         squeeze=False)
for i, source in enumerate(sources):
    for j, m in enumerate(DARK_METRICS):
        ax = axes[i, j]
        if m == "point_f1" and source != "spacenet":
            ax.axis("off")  # intersections are only labelled on SpaceNet
            continue
        for v in VARIANTS:
            sub = dark[v][dark[v]["source"] == source]
            means = sub.groupby("severity")[m].mean()
            if means.notna().any():
                ax.plot(means.index, means.values, marker="o", markersize=6, linewidth=2,
                        color=COLORS[v], label=LABELS[v])
        ax.set_title(f"{source}: {m}", fontsize=10)
        ax.set_xlabel("darkening severity")
        ax.grid(color="#e4e3dd", linewidth=0.8)
        ax.set_axisbelow(True)
axes[0, 0].legend(fontsize=8)
fig.suptitle("Mean metric vs. darkening severity, both models")
fig.tight_layout()
plt.show()

dark_table = pd.concat({v: dark[v].groupby(["source", "severity"])[DARK_METRICS[:3]].mean()
                        for v in VARIANTS}, axis=1)
display(dark_table.round(3))

## 6. Findings

*To be written once both jobs have run.* Record, per claim, what the tables show, including a
null or contrary result:

- **Metric claim:** do mIoU/pixel accuracy and PLEM's metrics move the same way between the two
  models (section 2), and how often do they disagree tile by tile (section 3)?
- **Loss claim:** which PLEM metrics, if any, favour the joint loss with a confidence interval
  excluding zero?
- **Intersections:** baseline at 0.5 vs. calibrated vs. the joint model.
- **Caveats:** one seed per model; the baseline's BCE `pos_weight` cap (100) and the joint model's
  heatmap settings were each set once, not tuned.